In [1]:
import os
from pathlib import Path
import json
import requests
import re
import math
import ipywidgets as widgets
from IPython.display import display
from dotenv import load_dotenv
from openai import OpenAI
#from ollama import chat
#from anthropic import Anthropic
from IPython.display import Markdown, display
from agents import Agent, Runner, WebSearchTool, function_tool, AgentOutputSchema, output_guardrail, input_guardrail
from agents.model_settings import ModelSettings
import asyncio
from pydantic import BaseModel
from typing import Dict, Any
import traceback 

In [2]:
for _d in [Path.cwd(), *Path.cwd().parents]:
    _env = _d / ".env"
    if _env.is_file():
        load_dotenv(_env, override=True)
        break

API_KEY_WEATHER = os.environ["WEATHERAPI_KEY"]
API_KEY_EXCHANGE = os.environ["EXCHANGE_RATE_API_KEY"]

In [3]:
ROUTER_PROMPT = """
You are a STRICT routing assistant.

Your ONLY job is to classify the user input.

You MUST choose exactly ONE intent:
- getWeather
- calculateMath
- getExchangeRate
- generalChat

ABSOLUTE RULES:
- No explanations
- No markdown
- No extra text
- Output ONLY JSON

Return ONLY valid JSON in this exact format:

{
  "intent": "<one of: calculateMath, getWeather, getExchangeRate, generalChat>",
  "parameters": {},
  "confidence": <float between 0 and 1>
}

Do not include any extra text.

EXAMPLES:

User: What's the weather in Paris?
{"intent":"getWeather","parameters":{"city":"Paris"},"confidence":0.95}

User: 5 + 3 * 2
{"intent":"calculateMath","parameters":{"expression":"5+3*2"},"confidence":0.98}

User: I had 5 apples, ate 2 and bought 10, how many now?
{"intent":"calculateMath","parameters":{"problem":"word_problem"},"confidence":0.9}

User: USD to ILS
{"intent":"getExchangeRate","parameters":{"from":"USD","to":"ILS"},"confidence":0.95}

User: Tell me a joke
{"intent":"generalChat","parameters":{},"confidence":0.8}

-------------------------
CLASSIFICATION RULES:
-------------------------

1. WEATHER:
- If input is a city name ONLY (e.g., "London", "Paris") → getWeather
- If input asks about weather → getWeather
- ANY comparison between cities involving temperature, weather, hotter, colder → MUST be getWeather

- Phrases like:
  "how much hotter"
  "how much colder"
  "difference between"
  "compare weather"
  "which is hotter"
  "temperature difference"

→ ALWAYS classify as getWeather

- Even if the word "weather" is NOT present

- Even if grammar is imperfect

For Example:
  User: how much hotter tel aviv then london
  {"intent":"getWeather","parameters":{"cities":["Tel Aviv","London"],"mode":"diff"},"confidence":0.95}

2. MATH:
- If input contains numbers + operators (+ - * /) → calculateMath
- If input is a math question or word problem → calculateMath

3. EXCHANGE RATE:
If the user input contains ANY currency name (e.g. euro, USD, dollar, shekel),
AND no other context is given,
→ classify as getExchangeRate
Examples:
User: euro to dollar
{"intent":"getExchangeRate","parameters":{"from":"euro","to":"dollar"},"confidence":0.95}
User: euro
{"intent":"getExchangeRate","parameters":{"from":"euro"},"confidence":0.9}

4. GENERAL CHAT:
- Everything else

"""

In [4]:
GET_WEATHER_SYSTEM_PROMPT = """
You are a weather agent.

Your job:

1. Extract ALL cities from the user input
2. Call getWeather with a list of cities
3. If multiple cities:
   - Compute result based on intent:

MODES:
- diff → absolute temperature difference
- ratio → how many times hotter
- compare → describe both

Examples:

1. User: "how much hotter is Tel Aviv than Tokyo"

Steps:
- cities = ["Tel Aviv", "Tokyo"]
- mode = "diff"
- call tool
- compute: Tel Aviv temp - Tokyo temp

Final answer:
"Tel Aviv is 3°C hotter than Tokyo"


2. User: "what is the weather in Tel Aviv"

Final answer:
"Tel Aviv is 30°C sunny with 10 km/h wind"

AFTER calling the tool:

- If mode = "diff":
  compute: difference = city1.temperature - city2.temperature
  return: "<city1> is X°C hotter/colder than <city2>"

- If mode = "compare":
  describe both temperatures

- Always produce a FINAL human-readable answer
- NEVER return raw tool output


"""

In [5]:
GET_EXCHANGE_SYSTEM_PROMPT = """
You are a currency exchange agent.

Your ONLY job is to extract and normalize parameters for the getExchangeRate tool.

You MUST follow these rules:

1. INPUT TYPES:

- If the user provides TWO currencies (e.g. "2 euros in usd"):
  → extract:
    from = source currency
    to = target currency
    amount = number (default 1 if missing)

- If the user provides ONLY ONE currency (e.g. "euro", "2 euros"):
  → ALWAYS assume:
    to = ILS
    from = detected currency
    amount = number (default 1 if missing)

2. NORMALIZATION RULES:
- Convert all currencies to standard 3-letter codes:
  euro → EUR
  usd → USD
  dollar → USD
  shekel → ILS

3. DEFAULT RULE:
If only one currency is mentioned:
→ to_currency MUST be "ILS"

4. OUTPUT FORMAT:
After calling the tool, return ONLY a single line like:
20 EUR = 74.6 ILS

No JSON. No explanations. Just the final answer.

5. IMPORTANT:
- Do NOT explain anything
- Do NOT add text
- Do NOT return extra fields
- Only return valid JSON
"""

In [6]:
CALCULATOR_SYSTEM_PROMPT = """
You are a calculator agent.

Your job is to solve arithmetic problems using the calculateMath tool.

-------------------------
WORKFLOW
-------------------------
1. Convert the user input into a valid arithmetic expression.
2. Call the calculateMath tool with that expression.
3. Take the result from the tool.
4. Return the final answer.

-------------------------
INPUT TYPES
-------------------------
You may receive:
- Direct expressions (e.g. "7 - 4", "2 * (3 + 5)", "10 / 2")
- Word problems involving basic arithmetic

For word problems:
- Extract only the necessary numbers and operation
- Convert to a single arithmetic expression
- Do NOT explain anything

Examples:
- "if I had 7 apples and ate 4" → 7 - 4
- "10 candies and gave 3 away" → 10 - 3
- "5 packs with 4 items each" → 5 * 4

-------------------------
TOOL USAGE
-------------------------
Call calculateMath exactly once with the final expression.

After receiving the tool result:
- Immediately return the final answer
- Do not call any tools again

-------------------------
OUTPUT RULES (VERY IMPORTANT)
-------------------------
After receiving tool result, output ONLY:
expression = result

Examples:
input: 7-4
output: 7 - 4 = 3
input: 10/2 = 5
output: 10 / 2 = 5
input: 5*4 = 20
output: 5 * 4 = 20

- Do NOT output anything else
- No explanations
- No extra text
- No tool output copying

-------------------------
CONSTRAINTS
-------------------------
- Only use +, -, *, /
- Ignore irrelevant words
- Never ask questions
- Never output intermediate steps
"""

In [7]:
GENERAL_CHAT_SYSTEM_PROMPT = """
You are a cynical but helpful research assistant.

Rules:
- Keep answers short
- Occasionally use data engineering metaphors
- Stay consistent in tone

Safety:
- Refuse political questions
- Refuse malicious code requests
- Refuse unsafe content

If unsafe:
Respond ONLY with:
"I cannot process this request due to safety protocols."
"""

In [8]:
class GuardrailResult:
    def __init__(self, tripwire_triggered, output):
        self.tripwire_triggered = tripwire_triggered
        self.output = output

In [9]:
@input_guardrail
def input_guardrails(user_input, context=None, agent=None): # there is an error in this fucntion
    text = getattr(user_input, "input", None)

    if text is None:
        text = str(user_input)

    blocked_keywords = ["hack", "attack", "malware", "politics"]
    if not text.strip():
        return GuardrailResult(True, "Input cannot be empty")

    if any(word in text.lower() for word in blocked_keywords):
        return GuardrailResult(
            True,
            "⚠️ I can't respond to that request.")

    return GuardrailResult(False, text)

In [10]:
@output_guardrail
def output_guardrails_chat(output, context=None, agent=None): # there is an error in this fucntion
    # 1. block unsafe content
    text = str(output)

    # ONLY enforce length (no blocking logic)
    if len(text.split()) > 50:
        return GuardrailResult(False, text[:200] + "...")

    return GuardrailResult(False, text)

In [11]:
def validate_router_output(route): # unused function, but shows how to validate router output if we were to use it
    if route["intent"] not in ["getWeather", "calculateMath", "getExchangeRate", "generalChat"]:
        raise Exception("Invalid intent")

    if not (0 <= route["confidence"] <= 1):
        raise Exception("Invalid confidence")

In [12]:
LOG_FILE = "execution_log.txt"

def log_interaction(user_input, response):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"You: {user_input}\n")
        f.write(f"Bot: {response}\n")
        f.write("\n")

def reset_log():
    if os.path.exists(LOG_FILE):
        os.remove(LOG_FILE)
                

In [13]:
HISTORY_FILE = "history.json"

def load_history():
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, "r", encoding="utf-8") as f:
                history = json.load(f)

            print("ברוך שובך 👋")
            return history

        except Exception:
            print("History file corrupted, starting fresh.")
            return []

    else:
        return []


def save_history(history):
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)


def reset_history():
    if os.path.exists(HISTORY_FILE):
        os.remove(HISTORY_FILE)
    return []

def render_history(output_box, history, is_returning=False):
        output_box.outputs=()  # Clear previous outputs
        text = ""
        if history and is_returning:
            text += "ברוך שובך 👋\n\nConversation history 👇\n\n"
        for message in history:
            label = "You" if message["role"] == "user" else "Bot"
            text += f"{label}: {message['content']}\n\n"
        output_box.value = text
        num_lines = text.count('\n') + 1
        line_height = 20  # px per line
        new_height = max(100, num_lines * line_height)
        output_box.layout.height = f"{new_height}px"
        output_box.send_state()

In [14]:
load_dotenv(override=True)
client = OpenAI()  

In [15]:
@function_tool
def calculateMath(expression: str):
    try:
        expression = expression.replace(" ", "")

        # only allow safe characters
        if not re.match(r'^[0-9+\-*/().]+$', expression):
            return "Invalid expression"

        result = eval(expression, {"__builtins__": None}, {})
        
        return str(result)

    except:
        return "Invalid expression"

In [16]:
def extract_amount(text):
    match = re.search(r'\d+(\.\d+)?', text)
    return float(match.group()) if match else None

In [17]:
def get_weather_description(code):
    if code == 0:
        return "Clear sky"
    elif code in [1, 2, 3]:
        return "Cloudy"
    elif code in [61, 63, 65]:
        return "Rain"
    elif code in [71, 73, 75]:
        return "Snow"
    else:
        return "Unknown"

In [18]:
@function_tool
def getWeather(cities: list[str]):
    results = []
    if not cities:
        return "Please provide at least one city"

    for city in cities:
        try:
            geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
            geo_response = requests.get(geo_url).json()

            if "results" not in geo_response or not geo_response["results"]:
                results.append(f"City not found: {city}")
                continue

            lat = geo_response["results"][0]["latitude"]
            lon = geo_response["results"][0]["longitude"]

            weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
            weather_response = requests.get(weather_url).json()

            if "current_weather" not in weather_response:
                continue

            temp = weather_response["current_weather"]["temperature"]
            wind = weather_response["current_weather"]["windspeed"]
            weather_code = weather_response["current_weather"]["weathercode"]

            description = get_weather_description(weather_code)

            results.append({
                "city": city,
                "temperature": temp
            })

        except Exception as e:
            results.append(f"Error for {city}: {str(e)}")

    return results

In [19]:
@function_tool
def getExchangeRate(fromCurrency: str, toCurrency: str = None):
    
    fromCurrency = fromCurrency.upper()
 
    if not toCurrency:
        toCurrency = "ILS"
    else:
        toCurrency = toCurrency.upper()

    url = f"https://v6.exchangerate-api.com/v6/{API_KEY_EXCHANGE}/latest/{fromCurrency}"

    try:
        response = requests.get(url)
        data = response.json()

        if data.get("result") != "success":
            return "API error"

        rate = data["conversion_rates"].get(toCurrency)

        if rate is None:
            return f"No data for {toCurrency}"

        return f"1 {fromCurrency} = {rate} {toCurrency}"

    except Exception as e:
        return f"Error: {str(e)}"

In [20]:
class RouterOutput(BaseModel):
    intent: str
    parameters: Dict[str, Any]
    confidence: float

In [21]:
router_agent = Agent(
    name="Router agent",
    instructions=ROUTER_PROMPT,
    tools=[],
    model="gpt-4o-mini",
    input_guardrails=[input_guardrails], # need to add the input Guardrail here, but it causes an error so I'm leaving it out for now
    output_type=AgentOutputSchema(RouterOutput, strict_json_schema=False)
)

In [22]:
math_agent = Agent(
    name="Math agent",
    instructions=CALCULATOR_SYSTEM_PROMPT,
    tools=[calculateMath],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
    output_type=str  
)

In [23]:
weather_agent = Agent(
    name="Weather agent",
    instructions=GET_WEATHER_SYSTEM_PROMPT,
    tools=[getWeather],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="auto"),
    output_type=str
)

In [24]:
currency_agent = Agent(
    name="Currency agent",
    instructions=GET_EXCHANGE_SYSTEM_PROMPT,
    tools=[getExchangeRate],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="auto"),
    output_type=str
)

In [25]:
general_chat_agent = Agent(
    name="general chat agent",
    instructions=GENERAL_CHAT_SYSTEM_PROMPT,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="auto"),
    output_guardrails=[output_guardrails_chat], # there is an error in the guardrail function causes to crash the agent. need to add 
    output_type=str
)

In [26]:
def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else "{}"

In [27]:
def extract_cities(user_input):
    # simple rule-based first (you can later replace with LLM)
    cities = []

    # very basic pattern ideas
    # split by "then", "than", "vs", "compared to"
    separators = [" then ", " than ", " vs ", " compared to "]

    text = user_input.lower()
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            cities = [parts[0], parts[1]]
            break

    return [c.strip() for c in cities]

In [28]:
def extract_temp(weather_result):
    nums = re.findall(r'[-+]?\d*\.?\d+', str(weather_result))

    if not nums:
        return None

    return float(nums[0])

In [29]:
def main():
    history = load_history()

    # -------------------------
    # WIDGETS
    # -------------------------
    output_box = widgets.Textarea(
        placeholder='💬 Messages...',
        layout=widgets.Layout(
           width='100%',
         height='320px'
        ),
        disabled=True
    )

    input_box = widgets.Text(
    placeholder='Ask something...',
    layout=widgets.Layout(width='70%')
    )

    send_button = widgets.Button(
        description='Send',
        button_style='primary',
        layout=widgets.Layout(width='100px')
    )

    input_area = widgets.HBox(
    [input_box, send_button],
    layout=widgets.Layout(justify_content='center', margin='10px 0')
    )

    ui = widgets.VBox(
    [output_box, input_area],
    layout=widgets.Layout(width='70%', margin='0 auto')
    )

    display(ui)

    # -------------------------
    # RENDER HISTORY
    # -------------------------
    def render_history():
        text = ""
        for msg in history:
            if msg["role"] == "user":
                text += f"\n🧑 You: {msg['content']}\n"
            else:
                text += f"🤖 Bot: {msg['content']}\n"

        output_box.value = text

    render_history()

    # -------------------------
    # MAIN LOGIC
    # -------------------------
    async def handle_message():
        nonlocal history

        user_input = input_box.value.strip()
        if not user_input:
            return

        input_box.value = ""

        # EXIT
        if user_input.lower() == "/exit":
            output_box.value += "\nGoodbye 👋\n"
            input_box.disabled = True
            return

        # RESET
        if user_input.lower() == "/reset":
            history = reset_history()
            render_history()
            return

        # ADD USER MESSAGE
        history.append({"role": "user", "content": user_input})
        render_history()

        try:
            route = await asyncio.wait_for(
                Runner.run(router_agent, user_input),
                timeout=5
            )

            intent = route.final_output.intent
            params = route.final_output.parameters

            agent_map = {
                "calculateMath": math_agent,
                "getWeather": weather_agent,
                "getExchangeRate": currency_agent,
                "generalChat": general_chat_agent
            }

            selected_agent = agent_map.get(intent, general_chat_agent)

            if intent == "calculateMath":
                expr = params.get("expression") or user_input
                response = await asyncio.wait_for(
                    Runner.run(selected_agent, expr),
                    timeout=20
                )
            else:
                response = await asyncio.wait_for(
                    Runner.run(selected_agent, user_input),
                    timeout=5
                )

            response_text = response.final_output or str(response)

        except Exception as e:
           
            error_text = str(e)

            if "Guardrail" in error_text:
                response_text = "⚠️ I can’t respond to that request."

            elif "timeout" in error_text.lower():
                response_text = "⏳ Request timed out. Try again."

            else:
                 response_text = f"❌ ERROR: {error_text}"

        # SAVE RESPONSE
        history.append({"role": "assistant", "content": response_text})
        save_history(history)

        render_history()

    # -------------------------
    # HANDLERS
    # -------------------------
    def run_async():
        loop = asyncio.get_event_loop()
        if loop.is_running():
            loop.create_task(handle_message())
        else:
            loop.run_until_complete(handle_message())

    send_button.on_click(lambda x: run_async())
    input_box.on_submit(lambda x: run_async())

In [30]:
main()        


ברוך שובך 👋


C:\Users\Yana Naydenova\AppData\Local\Temp\ipykernel_40636\3375675519.py:144: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(lambda x: run_async())
